# Physical Activity

In [1]:
%pip install -q -r ../../requirements.txt

# If this block is stuck: 
# Press: Ctrl + Shift + P
# Run: Developer: Reload Window

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
# from scipy.stats import entropy
import pandas as pd
import os
import numpy as np

In [ ]:
# Load CSV files
data_dir = '../../mcphases/'

active_minutes = pd.read_csv(os.path.join(data_dir, 'active_minutes.csv'))
calories = pd.read_csv(os.path.join(data_dir, 'calories.csv'))
demographic_vo2_max = pd.read_csv(os.path.join(data_dir, 'demographic_vo2_max.csv'))
# exercise = pd.read_csv(os.path.join(data_dir, 'exercise.csv'))
time_in_heart_rate_zones = pd.read_csv(os.path.join(data_dir, 'time_in_heart_rate_zones.csv'))
# height_and_weight = pd.read_csv(os.path.join(data_dir, 'height_and_weight.csv'))
subject_info = pd.read_csv(os.path.join(data_dir, 'subject-info.csv'))
hormones_and_selfreport = pd.read_csv(os.path.join(data_dir, 'hormones_and_selfreport.csv'))

print("All CSV files loaded successfully!")

All CSV files loaded successfully!


In [4]:
# Convert self-report symptoms to numeric values
likert_map = {
    'Not at all': 0,
    'Very Low/Little': 1,
    'Low': 2,
    'Moderate': 3,
    'High': 4,
    'Very High': 5
}

symptoms = [
    'appetite',
    'exerciselevel',
    'headaches',
    'cramps',
    'sorebreasts',
    'fatigue',
    'sleepissue',
    'moodswing',
    'stress',
    'foodcravings',
    'indigestion',
    'bloating'
]

for col in symptoms:
    hormones_and_selfreport[col + '_num'] = (
        hormones_and_selfreport[col]
        .map(likert_map)
    )

### Prepare the individual dataframes

To include from every dataframe: id, day_in_study, is_weekend

active_minutes: sedentary, lightly, moderately, very

calories: calories (daily sum based on day_in_study)

demographic_vo2_max: filtered_demographic_vo2_max

(ignore) exercise: start_day_in_study as day_in_study, 
    activitylevel (looks like: "[{'minutes': 0, 'name': 'sedentary'}, {'minutes': 3, 'name': 'lightly'}, {'minutes': 11, 'name': 'fairly'}, {'minutes': 2, 'name': 'very'}]"), 
    heartratezones (looks like: "[{'name': 'Out of Range', 'min': 30, 'max': 121, 'minutes': 16, 'caloriesOut': 69.75168}, {'name': 'Fat Burn', 'min': 121, 'max': 143, 'minutes': 0, 'caloriesOut': 0.0}, {'name': 'Cardio', 'min': 143, 'max': 171, 'minutes': 0, 'caloriesOut': 0.0}, {'name': 'Peak', 'min': 171, 'max': 220, 'minutes': 0, 'caloriesOut': 0.0}]")

time_in_heart_rate_zones: in_default_zone_3, in_default_zone_2, in_default_zone_1, below_default_zone_1

(ignore) height_and_weight: height_2022, height_2024, weight_2022, weight_2024 --> calculate BMI

subject_info: birth_year --> age, ethnicity, sexually_active, self_report_menstrual_health_literacy --> convert to numerical feature, age_of_first_menarche

hormones_and_selfreport 

#### Active minutes

In [10]:
active_minutes = active_minutes.drop(columns=["study_interval"])
active_minutes.head()

,id,is_weekend,day_in_study,sedentary,lightly,moderately,very
0,1,True,1,753.0,64,0,0
1,1,False,2,855.0,74,0,0
2,1,False,3,751.0,134,18,7
3,1,False,4,905.0,86,0,0
4,1,False,5,1430.0,10,0,0


#### Calories

In [7]:
daily_calories = (
    calories
    .groupby(["id", "day_in_study"], as_index=False)
    .agg(
        is_weekend=("is_weekend", "first"),
        calories_sum=("calories", "sum")
    )
)

daily_calories.head()

,id,day_in_study,is_weekend,calories_sum
0,1,1,True,1542.0
1,1,2,False,1591.0
2,1,3,False,1755.0
3,1,4,False,1552.0
4,1,5,False,1456.0


#### Demographic VO2 max

In [12]:
demographic_vo2_max = demographic_vo2_max.drop(columns=["study_interval", "demographic_vo2_max", 
                                                        "demographic_vo2_max_error", "filtered_demographic_vo2_max_error"])
demographic_vo2_max.head()

,id,is_weekend,day_in_study,filtered_demographic_vo2_max
0,1,True,1,33.79370
1,1,False,2,32.55987
2,1,False,3,31.50628
3,1,False,4,31.06774
4,1,False,5,30.88130


#### Exercise

In [21]:
print("Total rows:", len(exercise))
print("Exact duplicates:", exercise.duplicated().sum())

Total rows: 7282
Exact duplicates: 3641


In [22]:
exercise.groupby(list(exercise.columns)).size().value_counts()

1    2382
5     363
6     242
2     111
7      45
8      30
4       1
Name: count, dtype: int64

In [44]:
exercise_unique = exercise.drop_duplicates()
print("Exact duplicates for new dataframe:", exercise_unique.duplicated().sum())

Exact duplicates for new dataframe: 0


In [49]:
exercise_unique = exercise_unique.drop(columns=["study_interval", "last_modified_day_in_study", "last_modified_timestamp", "activitytypeid",
                                                "original_start_day_in_study", "original_start_timestamp", "originalduration", "duration",
                                                "manualvaluesspecified", "logtype", "hasgps", "shouldfetchdetails", "hasactivezoneminutes"])
exercise_unique = exercise_unique.rename(columns={"start_day_in_study": "day_in_study"})
exercise_unique.head(5)

,id,is_weekend,day_in_study,start_timestamp,activityname,activitylevel,averageheartrate,calories,activeduration,steps,heartratezones,activezoneminutes,elevationgain
0,1,False,39,21:23:14,Walk,"[{'minutes': 0, 'name': 'sedentary'}, {'minute...",106.0,96,1127000,1848.0,"[{'name': 'Out of Range', 'min': 30, 'max': 12...","{'totalMinutes': 1, 'minutesInHeartRateZones':...",18.288
1,1,False,39,21:23:14,Walk,"[{'minutes': 0, 'name': 'sedentary'}, {'minute...",106.0,96,1127000,1848.0,"[{'name': 'Out of Range', 'min': 30, 'max': 12...","{'totalMinutes': 1, 'minutesInHeartRateZones':...",18.288
7,1,False,44,19:56:44,Outdoor Bike,"[{'minutes': 24, 'name': 'sedentary'}, {'minut...",91.0,30,1639000,NaN,"[{'name': 'Out of Range', 'min': 30, 'max': 12...","{'totalMinutes': 0, 'minutesInHeartRateZones':...",0.000
10,1,False,44,18:33:58,Walk,"[{'minutes': 0, 'name': 'sedentary'}, {'minute...",100.0,268,3430000,5356.0,"[{'name': 'Out of Range', 'min': 30, 'max': 12...","{'totalMinutes': 0, 'minutesInHeartRateZones':...",3.454
14,1,False,44,18:33:58,Walk,"[{'minutes': 0, 'name': 'sedentary'}, {'minute...",100.0,268,3430000,5356.0,"[{'name': 'Out of Range', 'min': 30, 'max': 12...","{'totalMinutes': 0, 'minutesInHeartRateZones':...",3.454


#### Time in heart rate zones

In [62]:
time_in_heart_rate_zones = time_in_heart_rate_zones.drop(columns=["study_interval"])
time_in_heart_rate_zones.head()

,id,is_weekend,day_in_study,in_default_zone_3,in_default_zone_2,in_default_zone_1,below_default_zone_1
0,1,True,1,0.0,0.0,126.0,1036.0
1,1,False,2,5.0,82.0,416.0,512.0
2,1,False,3,5.0,119.0,599.0,368.0
3,1,False,4,0.0,0.0,212.0,613.0
4,1,False,5,8.0,123.0,250.0,308.0


In [64]:
time_in_heart_rate_zones = time_in_heart_rate_zones.rename(
    columns={"in_default_zone_3": "peak_zone", 
             "in_default_zone_2": "cardio_zone", 
             "in_default_zone_1": "fat_burn_zone", 
             "below_default_zone_1": "below_fat_burn_zone"})
time_in_heart_rate_zones.head()

,id,is_weekend,day_in_study,peak_zone,cardio_zone,fat_burn_zone,below_fat_burn_zone
0,1,True,1,0.0,0.0,126.0,1036.0
1,1,False,2,5.0,82.0,416.0,512.0
2,1,False,3,5.0,119.0,599.0,368.0
3,1,False,4,0.0,0.0,212.0,613.0
4,1,False,5,8.0,123.0,250.0,308.0


In [ ]:
time_in_heart_rate_zones["id"].unique()

array([ 1,  2,  3,  4,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 18, 19,
       20, 22, 23, 24, 26, 27, 29, 30, 32, 33, 34, 37, 38, 39, 40, 41, 42,
       43, 44, 45, 46, 47, 48, 49, 50])

#### Height and Weight

In [65]:
height_and_weight.head()

,id,height_2022,weight_2022,height_2024,weight_2024
0,1,NaN,NaN,NaN,NaN
1,2,NaN,NaN,NaN,NaN
2,3,167.0,52.0,NaN,NaN
3,4,170.0,78.0,NaN,NaN
4,6,NaN,NaN,NaN,NaN


In [ ]:
height_and_weight["height"] = height_and_weight[["height_2022", "height_2024"]].max(axis=1)
height_and_weight = height_and_weight.drop(columns=["height_2022", "height_2024"])
height_and_weight.head(100)

,id,weight_2022,weight_2024,height
0,1,NaN,NaN,NaN
1,2,NaN,NaN,NaN
2,3,52.0,NaN,167.0
3,4,78.0,NaN,170.0
4,6,NaN,NaN,NaN
5,7,56.0,NaN,170.0
6,8,NaN,NaN,NaN
7,9,50.8,53.5,160.0
8,10,60.3,NaN,161.0
9,11,NaN,NaN,NaN


In [98]:
height_and_weight["weight"] = (
    height_and_weight["weight_2024"]
    .fillna(height_and_weight["weight_2022"])
)

height_and_weight["BMI"] = (
    height_and_weight["weight"] /
    (height_and_weight["height"] / 100) ** 2
)

height_and_weight.head(50)

,id,weight_2022,weight_2024,height,BMI,weight
0,1,NaN,NaN,NaN,NaN,NaN
1,2,NaN,NaN,NaN,NaN,NaN
2,3,52.0,NaN,167.0,18.645344,52.0
3,4,78.0,NaN,170.0,26.989619,78.0
4,6,NaN,NaN,NaN,NaN,NaN
5,7,56.0,NaN,170.0,19.377163,56.0
6,8,NaN,NaN,NaN,NaN,NaN
7,9,50.8,53.5,160.0,20.898437,53.5
8,10,60.3,NaN,161.0,23.262991,60.3
9,11,NaN,NaN,NaN,NaN,NaN


In [103]:
print("Out of 42 participants,\n", height_and_weight["height"].isna().sum(), "participants have missing height data,\n", 
      height_and_weight["weight_2022"].isna().sum(), "participants have missing weight data in 2022,\n", 
      height_and_weight["weight_2024"].isna().sum(), "participants have missing weight data in 2024.")
total_missing = height_and_weight["height"].isna().sum() + height_and_weight["weight_2022"].isna().sum() + height_and_weight["weight_2024"].isna().sum()
print("That is a total of", total_missing, "missing values in the height and weight dataset.",
      "which is", (total_missing / (42 * 3) * 100), "% of the total data.")

Out of 42 participants,
 17 participants have missing height data,
 18 participants have missing weight data in 2022,
 31 participants have missing weight data in 2024.
That is a total of 66 missing values in the height and weight dataset. which is 52.38095238095239 % of the total data.


In [106]:
print("There are", height_and_weight["BMI"].isna().sum(), "missing BMI values, out of 42 participants," \
"which is", (height_and_weight["BMI"].isna().sum() / 42 * 100), "% of the total data.")

There are 18 missing BMI values, out of 42 participants,which is 42.857142857142854 % of the total data.


#### Subject info

In [ ]:
# subject_info: birth_year --> age
subject_info["age"] = 2024 - subject_info["birth_year"]
subject_info = subject_info.drop(columns=["birth_year", "gender", "education"])
subject_info.head()

,id,birth_year,gender,ethnicity,education,sexually_active,self_report_menstrual_health_literacy,age_of_first_menarche,age
0,1,1999,Woman,White,"Some university/ post-secondary, no degree",Yes,NaN,14,25
1,2,1995,Woman,East Asian,"Bachelor's degree (e.g. BA, BS)",Yes,High,13,29
2,3,2000,Woman,East Asian,"Bachelor's degree (e.g. BA, BS)",No,High,12,24
3,4,2000,Woman,South Asian,Doctorate or professional degree,No,Medium,12,24
4,6,1997,Woman,East Asian,Doctorate or professional degree,Yes,Low,13,27


In [116]:
# Convert self-report menstrual health literacy to numeric values
literacy_mapping = {
    "Non-existent": 0,
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Expert": 4
}

subject_info["self_report_menstrual_health_literacy_num"] = (
    subject_info["self_report_menstrual_health_literacy"]
    .map(literacy_mapping)
)
subject_info = subject_info.drop(columns=["self_report_menstrual_health_literacy"])
subject_info.head(100)

KeyError: 'self_report_menstrual_health_literacy'